In [1]:
import requests
from pathlib import Path
import pandas as pd
import io
from datetime import date
from zoneinfo import ZoneInfo
from datetime import datetime, timedelta, timezone
from collections import defaultdict
from utils import *

In [2]:
from dotenv import load_dotenv; load_dotenv()
import anthropic
client = anthropic.Anthropic()

resp = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=300,
    system="You are a concise tennis analyst.",
    messages=[{"role": "user", "content": "Explain what a walkover is in one sentence."}],
)
print(resp.content)       # a LIST of content blocks, not a string
print(resp.stop_reason)   # end_turn / max_tokens / pause_turn / tool_use
print(resp.usage)         # tokens = money; log this from day one

[TextBlock(citations=None, text='A walkover is a victory awarded to a player or team when their opponent fails to appear, withdraws, or is disqualified before the match begins, so the match is won without being played.', type='text')]
end_turn
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=36, output_tokens=55, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard')


In [3]:
SYSTEM = """You are a highly-skilled tennis analyst that researches pre-match tennis
news for a prediction dashboard.
Search for news from the last 7 days about each player in the given match.
Relevant: injuries, illness, withdrawals, retirements in recent matches, coaching changes,
travel/scheduling issues, surface or conditions comments, notable recent form.
For your final response generate a concise and informative summary of the information you have found,
including a final score-based verdict on who is better prepared for the match based on the information.

Structure:
- One report per player. The header for each report should be the player's name, nationality, age and current rank. You do not need to reference where you got this information. If you cannot find some part of this information for one of the players, do not list it for the other player. One bullet point each one 'Injury and Fitness', 'Last Match', 'Workload', 'Form and Mindset'. Do not add any extra bullet points or miss any out in this list. If you can't find any information for one of the bullet points, write 'No information found in this category'. Be honest: if you can't find anything, say that.
- Final summary. Give a concise sumary of the found information in one paragraph. In the second paragraph, give a verdict on each player's preparedness and whether this could shift the win likelihoods of the match.
Ignore: generic previews, betting tips, rankings recaps.
If nothing relevant exists, say so plainly — do not pad."""

In [4]:
def research_match(p1, p2, tournament, date):
    messages = [{"role" : "user",
                "content" : f"Match: {p1} vs {p2}, {tournament}, {date}. Find relevant news."}]

    while True:
        resp = client.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=2000,
            system=SYSTEM,
            messages=messages,
            tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 4}]
        )

        if resp.stop_reason != "pause_turn":
            break

        messages = messages.append({"role" : "assistant", "content" : resp.content})

    return resp

In [5]:
p1 = "Corentin Moutet"
p2 = "Felix Balshaw"
tournament = "Grand Prix Auvergne-Rhone-Alps"
date = "21/10/2026"

In [6]:
resp = research_match(p1, p2, tournament, date)

In [7]:
text = "".join(b.text for b in resp.content if b.type == "text")

In [13]:
print(text)

# Match report: Corentin Moutet vs Felix Balshaw (Grand Prix Auvergne-Rhône-Alpes, Lyon, 21/10/2026)

## Corentin Moutet – France, 27, No. 94
- **Injury and Fitness:** He was competing for the first time since retiring from his US Open opening match with a wrist injury. In the deciding set against Humbert he appeared to struggle with cramps and took a medical timeout for his wrist. When a journalist asked about the wrist a second time in his press conference, he answered tersely. He said he couldn't know whether it was fully healed.
- **Last Match:** He beat two-time defending champion Ugo Humbert 6-2, 6-7(5), 7-6(4). He won seven of the last nine points in the deciding tie-break. He saved two straight break points in the seventh game of the last set and spent the match running behind the baseline.
- **Workload:** The Humbert match lasted 3 hours 21 minutes, played on Tuesday, so he has only a short turnaround before this match. Before Lyon he had not played since the US Open, apart fr

In [14]:
def unpack(resp):
    text = "".join(b.text for b in resp.content if b.type == "text")
    sources = []
    for b in resp.content:
        if b.type == "web_search_tool_result" and isinstance(b.content, list):  # list = success
            sources += [{"url": r.url, "title": r.title, "page_age": r.page_age} for r in b.content]
    return text, sources, resp.usage.server_tool_use.web_search_requests

In [15]:
text, sources, servers = unpack(resp)

In [27]:
print(servers)

2


In [ ]:
from typing import Literal
from pydantic import BaseModel

class NewsItem(BaseModel):
    player: str
    category: Literal["injury", "withdrawal", "form", "coaching", "conditions", "other"]
    summary: str          # one sentence
    source_url: str
    published: str        # as stated by source, or "unknown"

class MatchNews(BaseModel):
    match_id: str
    has_relevant_news: bool
    headline: str         # one-line takeaway for the dashboard
    items: list[NewsItem]

def structure(match_id, research_text, sources):
    resp = client.messages.parse(
        model="claude-haiku-5-5",
        max_tokens=1500,
        output_format=MatchNews,
        messages=[{"role": "user", "content":
            f"match_id: {match_id}\n\nResearch notes:\n{research_text}\n\n"
            f"Available sources:\n{sources}\n\n"
            "Convert to the schema. Only use source_url values from the list above."}],
    )
    return resp.parsed_output

In [36]:
import anthropic
from dotenv import load_dotenv
import os
import requests
from datetime import datetime, timezone

load_dotenv()

today = datetime.now(timezone.utc).date().isoformat()
print(today)

base = "https://api.livetennisapi.com/api/public/v1"
HEADERS = {"X-API-Key": os.getenv("LIVE_TENNIS_API_KEY")}

re = requests.get(f"{base}/matches", headers=HEADERS, 
                  params={"status" : "upcoming",
                          "tour" : "atp",
                          "draw" : "singles",
                          "from" : f"{today}",
                          "to" : f"{today}"}).json()

2026-10-21


In [37]:
print(re["data"])

[{'court': None, 'created_at': '2026-10-18T19:31:48.437232Z', 'draw': 'singles', 'event_status': None, 'event_status_updated_at': None, 'format': 'BO3', 'gender': 'men', 'has_analysis': True, 'has_market': True, 'id': 206064, 'indoor': True, 'is_doubles': False, 'is_qualifying': False, 'live_at': None, 'outcome': None, 'players': {'p1': {'archive_player_id': 200329, 'backhand': 1, 'birthday': '1998-08-31', 'country': 'sui', 'data_completeness': {'known': 5, 'missing': [], 'of': 5}, 'hand': 'R', 'id': 93, 'is_doubles_team': False, 'itf_ranking': None, 'name': 'Remy Bertola', 'ranking': 219, 'ranking_checked_at': '2026-10-21T14:59:45.732938Z', 'ranking_movement': 'down', 'ranking_points': 256, 'ranking_status': 'ranked', 'tour': 'atp'}, 'p2': {'archive_player_id': 105676, 'backhand': 2, 'birthday': '1990-12-07', 'country': 'bel', 'data_completeness': {'known': 5, 'missing': [], 'of': 5}, 'hand': 'R', 'id': 523, 'is_doubles_team': False, 'itf_ranking': None, 'name': 'David Goffin', 'ranki

In [44]:
dicti = data_extractor(re["data"])

In [45]:
print(dicti)

[{'players': ('Remy Bertola', 'David Goffin'), 'birthdays': ('1998-08-31', '1990-12-07'), 'countries': ('sui', 'bel'), 'handedness': ('R', 'R'), 'rankings': (219, 657), 'points': (256, 53), 'tournament': 'Brussels', 'surface': 'hard', 'round': 'ATP Brussels - 1/16-finals', 'round_code': 'R32', 'is_indoor': True}, {'players': ('Moise Kouame', 'Lorenzo Musetti'), 'birthdays': ('2009-03-06', '2002-03-03'), 'countries': ('fra', 'ita'), 'handedness': ('R', 'R'), 'rankings': (194, 26), 'points': (300, 1870), 'tournament': 'Lyon (France)', 'surface': 'clay', 'round': 'ATP Lyon - 1/8-finals', 'round_code': 'R16', 'is_indoor': None}, {'players': ('Jan-Lennard Struff', 'Gauthier Onclin'), 'birthdays': ('1990-04-25', '2001-02-26'), 'countries': ('ger', 'bel'), 'handedness': ('R', 'R'), 'rankings': (58, 178), 'points': (929, 326), 'tournament': 'Brussels', 'surface': 'hard', 'round': 'ATP Brussels - 1/8-finals', 'round_code': 'R16', 'is_indoor': True}, {'players': ('Tommy Paul', 'Benjamin Bonzi'),